# Overview results

In [3]:
import pandas as pd

df = pd.read_csv("~/ownCloud/Personal/Structures/Democracy_Slaps/tweede-kamer/data/results/labled_full_gemma3_nl_nl.csv")

In [3]:
df.head()

,file_id,speaker_name,speaker_party,speech_text,jaar,date,kamer,category,title,url,...,vergadernummer,standardized_name,speech_text_en,final_label,duration,energy,raw,is_timeout,power-samples,result
0,h-tk-20142015-30-6,Mevrouw Schouten,CU,Uit de brief van de staatssecretaris begreep i...,2014-2015,27-11-2014,tk,handelingen,6 Begroting Sociale Zaken en Werkgelegenheid,https://zoek.officielebekendmakingen.nl/h-tk-2...,...,"nr. 30, item 6",schouten,From the letter from the State Secretary I und...,0,78.061899,2446.663314,"{'message': {'content': '```json\n{\n ""mentio...",False,"[29.15, 29.18, 28.98, 28.98, 28.83, 28.83, 28....","{'mentions': [{'name': 'staatssecretaris', 'ty..."
1,h-tk-20142015-50-8,Staatssecretaris Van Rijn,PvdA,In de motie-Pia Dijkstra op stuk nr. 118 worde...,2014-2015,4-2-2015,tk,handelingen,8 Noodscenario's voor pgb-houders,https://zoek.officielebekendmakingen.nl/h-tk-2...,...,"nr. 50, item 8",van van rijn,In the Pia Dijkstra motion on document no. 118...,0,236.759991,8712.795910,"{'message': {'content': '```json\n{\n ""mentio...",False,"[36.9, 37.14, 36.31, 36.22, 35.8, 35.8, 35.88,...","{'mentions': [{'name': 'Pia Dijkstra', 'type':..."
2,h-tk-20142015-12-4,Mevrouw Schouten,CU,Ik heb nog een vraag over het puntje waarom he...,2014-2015,9-10-2014,tk,handelingen,4 Bevorderen flexibel werken,https://zoek.officielebekendmakingen.nl/h-tk-2...,...,"nr. 12, item 4",schouten,I still have a question about why it is regula...,0,NaN,NaN,processing...,NaN,NaN,NaN
3,h-tk-20142015-23-6,Staatssecretaris Dijksma,PvdA,Nu wordt het lastig om te begrijpen wat de Kam...,2014-2015,12-11-2014,tk,handelingen,6 Verantwoorde groei melkveehouderij,https://zoek.officielebekendmakingen.nl/h-tk-2...,...,"nr. 23, item 6",dijksma,Now it is becoming difficult to understand wha...,0,121.705196,4358.329770,"{'message': {'content': '```json\n{\n ""mentio...",False,"[35.01, 35.01, 34.24, 34.24, 34.22, 34.22, 34....","{'mentions': [{'name': 'Schouw', 'type': 'pers..."
4,h-tk-20142015-72-29,De heer Van Oosten,VVD,Ik ga toch nog even terug naar de oorsprong va...,2014-2015,7-4-2015,tk,handelingen,29 Eigen bijdragen veroordeelden,https://zoek.officielebekendmakingen.nl/h-tk-2...,...,"nr. 72, item 29",van van oosten,I will go back to the origins of this bill: so...,0,NaN,NaN,processing...,NaN,NaN,NaN


In [4]:
print(f"len df original: {len(df)}")
df = df[df["raw"] != "processing..."]
df = df[df["is_timeout"] != True]
print(f"len df new: {len(df)}")

len df original: 390
len df new: 350


## Log data
Test execution start: May 18, 2025, 19:01:31

First forecsast: May 18, 2025, 19:27:02
* 📊 Forecast Report
* ✅ Finished: 200/390 (51.28%)
* ⏱️ Running since: 2025-05-18 19:01:31  Avg Duration: 75.95s
* ⏳ Estimated Remaining Time: 240.52 min
* ⚡ Estimated Total Energy: 954800.07 J
* 🤑 Estimated Cost: $0.34 💰

*** Gradually increase prediction 

Final Forecast: May 18, 2025, 20:57:31
* 📊 Forecast Report
* ✅ Finished: 390/390 (100.00%)
* ⏱️ Running since: 2025-05-18 19:01:31  Avg Duration: 121.09s
* ⏳ Remaining Time: N/A
* ⚡ Estimated Total Energy: 1470851.98 J
* 🤑 Estimated Cost: $9.41 💰

Cost beginning: 4972.49$ after run 4951.36$

* There were a few timeouts, the whole batch must have gotten deleted



## Results:

In [10]:
def clean_result(text: str):
        """
            Multiple ways to resolve different ways a llm might return json.
            1. normal
            2. embedded in text but marked as specified in the template
            3. normal json but with '' instead of ""
        """
        import json
        import ast
        import re
        text = str(text).strip()
        # Try to extract JSON block from markdown-style ```json ... ``` block
        if re.search(r'```json\b', text, re.IGNORECASE):
            try:
                match = re.search(r'```json\s*(.*?)\s*```', text, re.DOTALL | re.IGNORECASE)
                if match:
                    json_str = match.group(1).strip()
                    return json.loads(json_str)
            except Exception:
                pass

        # Try raw JSON
        try:
            return json.loads(text)
        except Exception:
            pass

        # Try using ast.literal_eval for single-quoted "JSON"
        try:
            return ast.literal_eval(text)
        except Exception:
            pass

        return None

In [12]:
df["result"]= df["result"].apply(clean_result)

In [15]:
df = df.dropna()

In [16]:
def compute_fallacy(data):
    return data["summary"]["count"] > 0

df["predicted"] = df["result"].apply(compute_fallacy)

In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
import seaborn as sns
import matplotlib.pyplot as plt
new_df = df
y_true = new_df["final_label"]
y_pred = new_df["predicted"]

# Use boolean labels for computation
labels = [False, True]
label_names = ["No Ad Hominem", "Ad Hominem"]

# Compute accuracy
accuracy = accuracy_score(y_true, y_pred)
precision_score = precision_score(new_df["final_label"], new_df["predicted"], zero_division=0)
recall_score= recall_score(new_df["final_label"], new_df["predicted"], zero_division=0)
f1_score = f1_score(new_df["final_label"], new_df["predicted"], zero_division=0)


print(f"Precision: {precision_score}")
print(f"Recall: {recall_score}")
print(f"F1 Score: {f1_score}")
print(f"✅ Accuracy: {accuracy:.2%}")

# Compute confusion matrix
cm = confusion_matrix(y_true, y_pred, labels=labels)

# Plot confusion matrix
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=label_names,
            yticklabels=label_names)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()

NameError: name 'new_df' is not defined